# 03. Основные эксперименты

В этом ноутбуке выполняется:
- обучение моделей на полном наборе признаков;
- перебор гиперпараметров;
- сравнение семейств моделей;
- эксперименты с PCA;
- выбор лучшей модели;
- оценка лучшей модели на test;
- сохранение артефактов.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.preprocessing import (
    load_raw_data,
    clean_recipes,
    clean_interactions,
    build_recipe_target,
    engineer_features,
    prepare_feature_sets,
    make_splits,
    save_processed_artifacts,
)

from src.modeling import (
    run_all_experiments,
    evaluate_best_model,
    save_model_artifacts,
)

from src.interpretation import (
    build_feature_groups,
    run_ablation_study,
    get_tree_feature_importance,
    get_permutation_importance_df,
    plot_top_features,
)


## Подготовка данных и полного набора признаков

In [2]:
recipes, interactions = load_raw_data("../data/raw")

recipes = clean_recipes(recipes)
interactions = clean_interactions(interactions)

data = build_recipe_target(recipes, interactions, min_rating_count=10)
data = engineer_features(data)

X_base, X_full, y, base_feature_cols, full_feature_cols = prepare_feature_sets(data)
splits = make_splits(X_base, X_full, y, random_state=42)

recipes duplicates by full row: 0
recipes duplicates by recipe_id: 0
interactions duplicates by full row: 0
raw thresholds: 4.625 4.857142857142857
rounded for report: 4.6 4.9
final recipe-level dataset shape: (19965, 17)

Распределение классов:
recipe_quality_name
normal    7917
bad       6044
good      6004

Доли классов:
recipe_quality_name
normal    0.3965
bad       0.3027
good      0.3007

Проверка диапазонов mean_rating по классам:
                     count     min     max    mean
recipe_quality_name                               
bad                   6044  2.1333  4.6250  4.4028
good                  6004  4.8571  5.0000  4.9348
normal                7917  4.6259  4.8559  4.7505
Количество столбцов после feature engineering: 52
                                name  recipe_id  minutes  contributor_id  submitted                                                                                                                                                                          

**Пояснение.** Для экспериментов используется полный набор из 38 признаков. Train/validation/test строятся один раз 
и дальше используются для всех моделей, чтобы сравнение экспериментов было честным. Все imputer/scaler/PCA-шаги 
находятся внутри sklearn Pipeline и обучаются только на train.


## Сохранение обработанных таблиц

In [3]:
summary_table, cleaning_table = save_processed_artifacts(
    data=data,
    recipes=recipes,
    interactions=interactions,
    y=y,
    processed_dir="../data/processed",
)

print("summary_table:")
print(summary_table.to_string(index=False))

print("\ncleaning_table:")
print(cleaning_table.to_string(index=False))

summary_table:
 rows  columns  class_0_bad_share  class_1_normal_share  class_2_good_share
19965       52             0.3027                0.3965              0.3007

cleaning_table:
 recipes_rows_after_cleaning  interactions_rows_after_cleaning  recipe_level_rows_after_target
                      230542                           1071520                           19965


## Полный запуск экспериментов

In [4]:
experiments, results_df, best_by_family_df = run_all_experiments(splits)

Всего экспериментов: 83


**Пояснение к экспериментам.** В этом блоке сравниваются несколько семейств моделей: Logistic Regression, KNN, 
RandomForest, ExtraTrees, HistGradientBoosting, а также PCA-варианты. Основной критерий выбора — `macro F1` на validation.


## Топ экспериментов на validation

In [5]:
print("15 лучших экспериментов:")
print(results_df.head(15).to_string(index=False))

15 лучших экспериментов:
              experiment       family feature_set  n_features                                                 params  macro_f1  weighted_f1  balanced_accuracy
  RF_full_d20_l10_mfsqrt RandomForest        full          38   max_depth=20, max_features=sqrt, min_samples_leaf=10    0.4217       0.4238             0.4217
   RF_full_d20_l10_mf0.7 RandomForest        full          38    max_depth=20, max_features=0.7, min_samples_leaf=10    0.4207       0.4232             0.4205
  RF_full_d10_l10_mfsqrt RandomForest        full          38   max_depth=10, max_features=sqrt, min_samples_leaf=10    0.4165       0.4151             0.4212
   RF_full_d20_l5_mfsqrt RandomForest        full          38    max_depth=20, max_features=sqrt, min_samples_leaf=5    0.4160       0.4216             0.4163
   RF_full_d10_l3_mfsqrt RandomForest        full          38    max_depth=10, max_features=sqrt, min_samples_leaf=3    0.4148       0.4144             0.4175
RF_full_dNone_l10_mfs

**Вывод по топу экспериментов.** Лучшие результаты показывает семейство RandomForest. Это ожидаемо: деревья хорошо 
работают с нелинейными зависимостями и признаками разного масштаба. Линейные модели и KNN уступают, а PCA-варианты 
не дают прироста качества.


## Лучшие модели по семействам

In [6]:
print(best_by_family_df.to_string(index=False))

             experiment       family feature_set  n_features                                               params  macro_f1  weighted_f1  balanced_accuracy
 RF_full_d20_l10_mfsqrt RandomForest        full          38 max_depth=20, max_features=sqrt, min_samples_leaf=10    0.4217       0.4238             0.4217
  ET_full_d20_l3_mfsqrt   ExtraTrees        full          38  max_depth=20, max_features=sqrt, min_samples_leaf=3    0.4070       0.4100             0.4069
  HGB_full_lr0.1_d8_l40       HistGB        full          38  learning_rate=0.1, max_depth=8, min_samples_leaf=40    0.4061       0.4172             0.4137
  KNN_full_k35_distance          KNN        full          38                     n_neighbors=35, weights=distance    0.3886       0.4004             0.3958
       LogReg_full_C1.0       LogReg        full          38                                                C=1.0    0.3780       0.3720             0.3940
     KNN_full_PCA_5_k15      KNN_PCA        full          38 n_n

**Вывод по семействам моделей.** Лучшей моделью стала RandomForest, за ней идут ExtraTrees и HistGradientBoosting. 
Это подтверждает, что ансамбли деревьев лучше подходят для текущего набора табличных признаков, чем простые baseline-модели.


## Оценка лучшей модели на test

In [7]:
best_exp, best_model_fitted, best_feature_cols = evaluate_best_model(
    experiments=experiments,
    results_df=results_df,
    splits=splits,
    base_feature_cols=base_feature_cols,
    full_feature_cols=full_feature_cols,
)

BEST EXPERIMENT ON VAL: RF_full_d20_l10_mfsqrt

RF_full_d20_l10_mfsqrt_TEST 
Macro F1: 0.4189
Weighted F1: 0.4199
Balanced accuracy: 0.4188

Classification report:
              precision    recall  f1-score   support

         bad     0.4248    0.3925    0.4080       907
      normal     0.4257    0.4322    0.4289      1187
        good     0.4086    0.4317    0.4199       901

    accuracy                         0.4200      2995
   macro avg     0.4197    0.4188    0.4189      2995
weighted avg     0.4203    0.4200    0.4199      2995

Confusion matrix:
             pred_bad  pred_normal  pred_good
true_bad          356          339        212
true_normal       323          513        351
true_good         159          353        389

Лучшая модель:
experiment: RF_full_d20_l10_mfsqrt
family: RandomForest
feature_set: full
params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 10}


**Вывод по финальной модели.** Финальная модель выбирается по validation, затем проверяется на test. 
В предыдущем прогоне лучшая конфигурация RandomForest дала `macro F1 = 0.4217` на validation и `macro F1 = 0.4189` 
на test. Небольшая разница между validation и test говорит о том, что модель не переобучилась сильно.


## Ablation study

Проверяем, какие группы признаков дают основной вклад в качество. Для этого обучаем финальную модель на разных 
поднаборах признаков: полный набор, без nutrition-признаков, без текстово-структурных признаков, без дат и на отдельных 
группах признаков.


In [ ]:
feature_groups = build_feature_groups(full_feature_cols)

ablation_df = run_ablation_study(
    model=best_exp["model"],
    splits=splits,
    feature_groups=feature_groups,
)

ablation_df


In [ ]:
ablation_df.to_csv("../data/processed/ablation_results.csv", index=False, encoding="utf-8")


**Вывод по ablation study.** Если качество заметно падает при исключении группы признаков, значит эта группа важна 
для модели. Этот блок нужен не только для улучшения качества, но и для интерпретации результата: он показывает, 
какие типы характеристик рецепта реально помогают классификации.


## Feature importance

Для финальной RandomForest-модели смотрим встроенную важность признаков. Это позволяет понять, какие признаки чаще 
используются деревьями при разбиениях.


In [ ]:
tree_importance_df = get_tree_feature_importance(
    model=best_model_fitted,
    feature_cols=best_feature_cols,
    top_n=20,
)

tree_importance_df


In [ ]:
plot_top_features(
    df=tree_importance_df,
    feature_col="feature",
    value_col="importance",
    title="Top-20 feature importances for RandomForest",
    output_path="../report/images/tree_feature_importance.png",
)


## Permutation importance

Дополнительно считаем permutation importance на validation. В отличие от встроенной важности RandomForest, 
этот метод оценивает падение качества при случайном перемешивании конкретного признака.


In [ ]:
permutation_importance_df = get_permutation_importance_df(
    model=best_model_fitted,
    X=splits["X_val_full"][best_feature_cols],
    y=splits["y_val"],
    feature_cols=best_feature_cols,
    top_n=20,
    n_repeats=5,
    random_state=42,
)

permutation_importance_df


In [ ]:
plot_top_features(
    df=permutation_importance_df,
    feature_col="feature",
    value_col="importance_mean",
    title="Top-20 permutation importances for RandomForest",
    output_path="../report/images/permutation_importance.png",
)


In [ ]:
tree_importance_df.to_csv("../data/processed/tree_feature_importance.csv", index=False, encoding="utf-8")
permutation_importance_df.to_csv("../data/processed/permutation_importance.csv", index=False, encoding="utf-8")


**Вывод по интерпретируемости.** Feature importance и permutation importance используются как объяснение финальной 
модели. Эти таблицы и графики нужно затем перенести в отчёт: они показывают, на какие характеристики рецепта модель 
опирается при предсказании качества.


## Сохранение модели и таблиц результатов

In [8]:
save_model_artifacts(
    results_df=results_df,
    best_by_family_df=best_by_family_df,
    best_model_fitted=best_model_fitted,
    best_feature_cols=best_feature_cols,
    processed_dir="../data/processed",
    models_dir="../models",
)

**Итог экспериментов.** В ноутбуке зафиксированы результаты перебора моделей, выбор финальной конфигурации, 
test-оценка, ablation study и интерпретация признаков. Эти результаты дальше используются в README и `report/report.md`.
